In [ ]:
!pip install -q -U transformers accelerate requests

import os
import re
import json
import glob
import torch
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_NAME = "OpenGVLab/InternVL3-8B-hf"

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
rms_eps = model.config.text_config.rms_norm_eps
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim,", model.config.text_config.num_key_value_heads, "kv heads")

In [ ]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

In [ ]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)
    inputs.pop("token_type_ids", None)
    return inputs


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [ ]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
TYPES = ["attribute", "relation"]
CAND_IDS = [9454, 2753, 9693, 2152]

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
expected = {
    t: {q["question_id"] for q in all_questions if q["hallucination_type"] == t and q["answer_format"] == "yes_no"}
    for t in TYPES
}
print(len(all_questions), "questions loaded,", {t: len(v) for t, v in expected.items()}, "yes/no questions")
print(processor.tokenizer.convert_ids_to_tokens(CAND_IDS))
assert processor.tokenizer.convert_ids_to_tokens(CAND_IDS) == ["Yes", "No", "yes", "no"]

with open(f"{INPUT}/vlm-labeled-examples/labeled_examples.json") as f:
    labels = pd.DataFrame(json.load(f))
assert "cache" in labels.columns, "attach vlm-labeled-examples version 3"
split_of = labels[labels["cache"] == "question_aware"].set_index("question_id")["split"].to_dict()
print(len(split_of), "questions with a split")

In [ ]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


def cache_files(t):
    files = sorted(
        glob.glob(f"/kaggle/input/datasets/*/question-aware-cache-internvl3-{t}/{t}_batch_*.pt"),
        key=batch_number,
    )
    assert files, f"attach question-aware-cache-internvl3-{t}"
    return files


records = []
for t in TYPES:
    files = cache_files(t)
    for path in files:
        batch = torch.load(path, mmap=True)
        for e in batch:
            records.append({
                "question_id": e["question_id"],
                "type": t,
                "prefix": e["question_id"].split("_")[0] + "_",
                "top_id": int(e["answer_logits"].argmax()),
                "answer_text": e["answer_text"],
                "n_image_tokens": e["n_image_tokens"],
                "cand": e["answer_logits"][CAND_IDS].float().tolist(),
            })
        del batch
    ids = {r["question_id"] for r in records if r["type"] == t}
    print(f"{t}: {len(files)} files, {len(ids)} entries, coverage {len(ids & expected[t]) / len(expected[t]):.4f}")
    assert len(ids & expected[t]) > 0.95 * len(expected[t]), f"{t} cache looks incomplete"

rec = pd.DataFrame(records)
rec = rec.drop_duplicates("question_id").reset_index(drop=True)

PAIRS = {}
for prefix in ["amber_", "reefknot_"]:
    sub = rec[rec.prefix == prefix]
    lower = sub.answer_text.str.strip().str.lower()
    yes_id = Counter(sub[lower == "yes"].top_id).most_common(1)[0][0]
    no_id = Counter(sub[lower == "no"].top_id).most_common(1)[0][0]
    PAIRS[prefix] = (yes_id, no_id)
    yes_purity = (sub[lower == "yes"].top_id == yes_id).mean()
    no_purity = (sub[lower == "no"].top_id == no_id).mean()
    print(f"{prefix:10s} yes_id={yes_id} {processor.tokenizer.decode(yes_id)!r} purity={yes_purity:.4f} | "
          f"no_id={no_id} {processor.tokenizer.decode(no_id)!r} purity={no_purity:.4f}")
    assert yes_id in CAND_IDS and no_id in CAND_IDS, "pair outside the stored candidate ids"
    assert yes_purity > 0.9 and no_purity > 0.9, f"mixed casing for {prefix}"

rec["gap"] = [
    c[CAND_IDS.index(PAIRS[p][0])] - c[CAND_IDS.index(PAIRS[p][1])] for c, p in zip(rec.cand, rec.prefix)
]
rec["in_pair"] = [t in PAIRS[p] for t, p in zip(rec.top_id, rec.prefix)]
rec["source"] = rec.prefix.str.rstrip("_")
rec["truth"] = rec.question_id.map(lambda q: question_by_id[q]["ground_truth_answer"].strip().lower())
rec["model_answer"] = np.where(rec.gap > 0, "yes", "no")
rec["group"] = np.select(
    [
        (rec.truth == "yes") & (rec.model_answer == "yes"),
        (rec.truth == "no") & (rec.model_answer == "no"),
        (rec.truth == "no") & (rec.model_answer == "yes"),
        (rec.truth == "yes") & (rec.model_answer == "no"),
    ],
    ["true_yes", "true_no", "false_yes", "false_no"],
    default="other",
)
rec["split"] = rec.question_id.map(split_of)
print(rec.split.isna().sum(), "cached questions without a split")
rec = rec[rec.split.notna()].reset_index(drop=True)

print(rec.groupby(["type", "source"]).in_pair.agg(["mean", "size"]).round(4))
print(pd.crosstab([rec.type, rec.source, rec.in_pair], rec.group))
print("image tokens:", rec.n_image_tokens.describe()[["min", "50%", "max"]].to_dict())
print("non-pair top-1 tokens:", Counter(
    processor.tokenizer.decode(i) for i in rec[~rec.in_pair].top_id
).most_common(8))

In [ ]:
TARGET = 3000
MIN_BUDGET = 1000
N_NONPAIR = 150
SHARES = {"true_yes": 0.4, "true_no": 0.4, "false_no": 0.2}

parts = []
for t in TYPES:
    d = rec[(rec.type == t) & rec.in_pair]
    fy = d[d.group == "false_yes"]
    budget = max(TARGET - len(fy), MIN_BUDGET)
    parts.append(fy.assign(stratum="false_yes"))
    for g, share in SHARES.items():
        pool = d[d.group == g]
        if len(pool) == 0:
            continue
        quota = min(int(budget * share), len(pool))
        parts.append(pool.groupby(["source", "split"], group_keys=False).sample(frac=quota / len(pool), random_state=0).assign(stratum=g))
    pool = rec[(rec.type == t) & ~rec.in_pair]
    parts.append(pool[pool.group == "false_yes"].assign(stratum="non_pair_false_yes"))
    rest = pool[pool.group != "false_yes"]
    parts.append(rest.sample(n=min(N_NONPAIR, len(rest)), random_state=0).assign(stratum="non_pair"))

selection = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
print(pd.crosstab([selection.type, selection.source, selection.split], selection.stratum, margins=True))
sel_counts = selection.groupby("type").size().to_dict()
print("selected per type:", sel_counts, "of", {t: len(expected[t]) for t in TYPES})

In [ ]:
_state = {"prefix": None}
_u_cache = {}

U = {
    prefix: ln_f.weight.float().cpu() * (lm_head.weight[yes_id] - lm_head.weight[no_id]).float().cpu()
    for prefix, (yes_id, no_id) in PAIRS.items()
}


def u_on(device):
    key = (_state["prefix"], device)
    if key not in _u_cache:
        _u_cache[key] = U[_state["prefix"]].to(device)
    return _u_cache[key]


HEAD_PROJ = {}
with torch.no_grad():
    for prefix in PAIRS:
        _state["prefix"] = prefix
        HEAD_PROJ[prefix] = []
        for layer in layers:
            w = layer.self_attn.o_proj.weight
            HEAD_PROJ[prefix].append(
                w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device)
            )

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * HEAD_PROJ[_state["prefix"]][l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + rms_eps)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

In [ ]:
@torch.no_grad()
def dla_question(image_path, question_text, prefix):
    _state["prefix"] = prefix
    yes_id, no_id = PAIRS[prefix]
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(**inputs)
    last_logits = outputs.logits[0, -1].float().cpu()

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[yes_id] - last_logits[no_id],
        "top_id": int(last_logits.argmax()),
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
        "n_image_tokens": int((inputs["input_ids"][0] == model.config.image_token_id).sum()),
    }

In [ ]:
import time

SOURCES = [("attribute/amber", "attribute", "amber_"), ("relation/amber", "relation", "amber_"), ("relation/reefknot", "relation", "reefknot_")]
N_PER_SOURCE = 16

results = {label: [] for label, _, _ in SOURCES}
rows = []
t0 = time.time()
for label, t, prefix in SOURCES:
    picked = rec[(rec.type == t) & (rec.prefix == prefix)].head(N_PER_SOURCE)
    assert len(picked) == N_PER_SOURCE, label
    for _, e in picked.iterrows():
        q = question_by_id[e.question_id]
        image_path = resolve_image_path(q)
        assert image_path is not None and os.path.exists(image_path), f"missing image for {e.question_id}"
        yes_id, no_id = PAIRS[prefix]
        r = dla_question(image_path, q["question_text"], prefix)
        total = r["embed"] + r["attn"].sum() + r["mlp"].sum()
        scale = r["embed"].abs() + r["attn"].abs().sum() + r["mlp"].abs().sum()
        if e.top_id == yes_id:
            sign_ok = bool(r["logit_diff"] > 0)
        elif e.top_id == no_id:
            sign_ok = bool(r["logit_diff"] < 0)
        else:
            sign_ok = True
        rows.append({
            "source": label,
            "rel_err_resid": ((total - r["resid_dot"]).abs() / scale).item(),
            "err_logit": (total / r["rms"] - r["logit_diff"]).abs().item(),
            "err_cache": abs(r["logit_diff"].item() - e.gap),
            "answer_match": r["answer_text"] == e.answer_text,
            "sign_ok": sign_ok,
            "in_pair": bool(e.in_pair),
            "tokens_match": r["n_image_tokens"] == e.n_image_tokens,
            "finite": bool(torch.isfinite(total) and torch.isfinite(r["logit_diff"])),
        })
        results[label].append(r)
sec_per_question = (time.time() - t0) / len(rows)
print(len(rows), "questions run,", f"{sec_per_question:.2f} s/question")

In [ ]:
n = len(rows)
for label in results:
    sub = [r for r in rows if r["source"] == label]
    print(f"{label:18s} rel err resid {max(r['rel_err_resid'] for r in sub):.5f} | "
          f"err logit {max(r['err_logit'] for r in sub):.4f} | err cache {max(r['err_cache'] for r in sub):.4f} | "
          f"answers match {sum(r['answer_match'] for r in sub)}/{len(sub)} | "
          f"sign ok {sum(r['sign_ok'] for r in sub)}/{len(sub)} | image tokens match {sum(r['tokens_match'] for r in sub)}/{len(sub)} | "
          f"top-1 in pair {sum(r['in_pair'] for r in sub)}/{len(sub)}")

assert all(r["finite"] for r in rows), "non-finite values"
assert max(r["rel_err_resid"] for r in rows) < 0.05, "components do not sum to the final residual"
assert max(r["err_logit"] for r in rows) < 0.5, "components do not reproduce the logit difference"
assert max(r["err_cache"] for r in rows) < 0.25, "prompt or token ids differ from the question-aware cache"
assert sum(r["answer_match"] for r in rows) == n, "answers differ from the question-aware cache"
assert all(r["sign_ok"] for r in rows), "logit difference sign disagrees with the cached answer"
assert all(r["tokens_match"] for r in rows), "image tiling differs from the question-aware cache"
print("validation passed")

hours = len(selection) * sec_per_question / 3600
print(f"selected {len(selection)} questions at {sec_per_question:.2f} s/question: about {hours:.1f} h of collection")

In [ ]:
for label, rs in results.items():
    attn = torch.stack([r["attn"] / r["rms"] for r in rs]).mean(0).flatten()
    mlp = torch.stack([r["mlp"] / r["rms"] for r in rs]).mean(0)
    mean_diff = torch.stack([r["logit_diff"] for r in rs]).mean().item()
    print(f"--- {label}: mean logit diff {mean_diff:.3f} ---")
    for i in attn.abs().topk(5).indices.tolist():
        print(f"  layer {i // n_head:2d} ({(i // n_head) / n_layer:.0%} depth) head {i % n_head:2d}  {attn[i]:+.3f}")
    for i in mlp.abs().topk(3).indices.tolist():
        print(f"  layer {i:2d} ({i / n_layer:.0%} depth) mlp      {mlp[i]:+.3f}")

In [ ]:
USERNAME = "nocturnalnerd18"
SEL_DIR = "/kaggle/working/dla_intern_selection"
os.makedirs(SEL_DIR, exist_ok=True)

with open(os.path.join(SEL_DIR, "selection.json"), "w") as f:
    json.dump({
        "pairs": PAIRS,
        "cand_ids": CAND_IDS,
        "selected": {t: selection[selection.type == t].question_id.tolist() for t in TYPES},
        "stratum": dict(zip(selection.question_id, selection.stratum)),
        "sec_per_question": sec_per_question,
    }, f)

with open(os.path.join(SEL_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "dla-intern-selection",
        "id": f"{USERNAME}/dla-intern-selection",
        "subtitle": "Question subsample for InternVL3 direct logit attribution",
        "description": "Questions selected for InternVL3-8B direct logit attribution from the question-aware caches: every false_yes row whose "
                       "top-1 is the yes/no pair, a stratified sample of true_yes, true_no and false_no by source and split, every false_yes row "
                       "whose top-1 is outside the pair plus a small sample of the other non-pair rows for a sensitivity check. stratum maps each question_id to its group.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

!kaggle datasets create -p {SEL_DIR}